In [45]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.cluster import KMeans

In [2]:
PATH = 'data/Omnichannel_Shopify_Sample.xlsx'
sheet_names = pd.ExcelFile(PATH).sheet_names
print(sheet_names)

['Orders', 'Order_Line_Items', 'Customers', 'Products', 'Inventory', 'Discounts', 'Returns', 'Google_Analytics', 'Meta_Ads', 'Email', 'Google_Ads', 'Display', 'Attribution']


In [16]:
data = pd.ExcelFile(PATH)
orders = pd.read_excel(data, sheet_name='Orders')
order_line_items = pd.read_excel(data, sheet_name='Order_Line_Items')
customers = pd.read_excel(data, sheet_name='Customers')
products = pd.read_excel(data, sheet_name='Products')
inventory = pd.read_excel(data, sheet_name='Inventory')
discounts = pd.read_excel(data, sheet_name='Discounts')
returns = pd.read_excel(data, sheet_name='Returns')
ga4 = pd.read_excel(data, sheet_name='Google_Analytics')
meta = pd.read_excel(data, sheet_name='Meta_Ads')
email = pd.read_excel(data, sheet_name='Email')
ga = pd.read_excel(data, sheet_name='Google_Ads')
display = pd.read_excel(data, sheet_name='Display')
attrib = pd.read_excel(data, sheet_name='Attribution')


In [17]:
attrib

,OrderID,CustomerID,SessionID,FirstTouch,LastTouch,Journey,Truth
0,100001,C10390,S498078,Google,Direct,Meta>Email,Meta40 Email60
1,100002,C10336,S970380,Google,Google,Google>Direct,Meta40 Email60
2,100003,C10087,S566549,Meta,Email,Meta>Email,Google70 Direct30
3,100004,C10353,S598614,Meta,Google,Meta>Email,Google70 Direct30
4,100005,C10458,S359311,Google,Google,Google>Direct,Meta40 Email60
...,...,...,...,...,...,...,...
1035,101036,C10040,S331430,Organic,Direct,Google>Direct,Meta40 Email60
1036,101037,C10429,S216959,Google,Direct,Google>Direct,Google70 Direct30
1037,101038,C10258,S140591,Organic,Direct,Meta>Email,Google70 Direct30
1038,101039,C10210,S398766,Meta,Google,Google>Direct,Meta40 Email60


In [18]:
order_line_items

,OrderID,SKU,Product,Qty,UnitPrice,Discount
0,100001,CAP003,Cap,3,63.57,6.98
1,100002,CAP003,Cap,1,57.92,14.58
2,100003,HOD015,Hoodie,3,116.18,14.77
3,100004,CAP003,Cap,2,74.10,2.70
4,100005,CAP003,Cap,2,119.67,2.03
...,...,...,...,...,...,...
1035,101036,DRS010,Summer Dress,2,105.02,13.47
1036,101037,DRS010,Summer Dress,3,29.39,8.96
1037,101038,JNS021,Slim Jeans,1,63.79,3.99
1038,101039,DRS010,Summer Dress,2,72.91,7.37


In [19]:
total_sales = orders.groupby('CustomerID')['TotalSales'].sum().to_frame()
total_sales

,TotalSales
CustomerID,
C10001,553.87
C10002,243.14
C10003,169.22
C10004,439.04
C10005,129.11
...,...
C10496,84.19
C10497,133.22
C10498,35.44


In [20]:
total_sales = total_sales.sort_values('TotalSales', ascending=False).reset_index()
total_sales['CumShare'] = ((total_sales['TotalSales'].cumsum()/sum(total_sales['TotalSales']))*100).round(2)
total_sales['CustomerShare'] = ((total_sales['TotalSales']/sum(total_sales['TotalSales']))*100).round(2)
total_sales

,CustomerID,TotalSales,CumShare,CustomerShare
0,C10098,1321.47,0.87,0.87
1,C10065,1210.02,1.67,0.80
2,C10273,1121.58,2.41,0.74
3,C10489,998.21,3.06,0.66
4,C10217,993.46,3.72,0.65
...,...,...,...,...
427,C10498,35.44,99.93,0.02
428,C10259,30.98,99.95,0.02
429,C10303,30.37,99.97,0.02
430,C10385,27.54,99.98,0.02


In [21]:
total_orders = orders.groupby(['CustomerID'])['OrderID'].count()
total_sales['NumOrders'] = total_sales['CustomerID'].map(total_orders)
total_sales['AvgSpendPerOrder'] = total_sales['TotalSales']/total_sales['NumOrders']
total_sales

,CustomerID,TotalSales,CumShare,CustomerShare,NumOrders,AvgSpendPerOrder
0,C10098,1321.47,0.87,0.87,6,220.245000
1,C10065,1210.02,1.67,0.80,5,242.004000
2,C10273,1121.58,2.41,0.74,6,186.930000
3,C10489,998.21,3.06,0.66,6,166.368333
4,C10217,993.46,3.72,0.65,5,198.692000
...,...,...,...,...,...,...
427,C10498,35.44,99.93,0.02,1,35.440000
428,C10259,30.98,99.95,0.02,1,30.980000
429,C10303,30.37,99.97,0.02,1,30.370000
430,C10385,27.54,99.98,0.02,1,27.540000


In [22]:
average_purchase_value = sum(total_sales['TotalSales'])/total_sales['TotalSales'].count()
avg_order_size = (total_sales['AvgSpendPerOrder']).mean()
avg_order_size_top_50 = total_sales[total_sales['CumShare']<51]['AvgSpendPerOrder'].mean()
avg_order_size_top_25 = total_sales[total_sales['CumShare']<26]['AvgSpendPerOrder'].mean()
avg_order_size_top_10 = total_sales[total_sales['CumShare']<11]['AvgSpendPerOrder'].mean()
avg_purchase_top_50 = total_sales[total_sales['CumShare']<51]['TotalSales'].mean()
avg_purchase_top_25 = total_sales[total_sales['CumShare']<26]['TotalSales'].mean()
avg_purchase_top_10 = total_sales[total_sales['CumShare']<11]['TotalSales'].mean()


In [23]:
print("Avg Spend per Customer: $", round(average_purchase_value,2))
print("Avg Spend per Customer - Top 50%: $", round(avg_purchase_top_50,2))
print("Avg Spend per Customer - Top 25%: $", round(avg_purchase_top_25,2))
print("Avg Spend per Customer - Top 10%: $", round(avg_purchase_top_10,2))
print("Avg Spend per Order: $", round(avg_order_size,2))
print("Avg Spend per Order - Top 50%: $", round(avg_order_size_top_50,2))
print("Avg Spend per Order - Top 25%: $", round(avg_order_size_top_25,2))
print("Avg Spend per Order - Top 10%: $", round(avg_order_size_top_10,2))


Avg Spend per Customer: $ 351.56
Avg Spend per Customer - Top 50%: $ 652.84
Avg Spend per Customer - Top 25%: $ 785.62
Avg Spend per Customer - Top 10%: $ 938.42
Avg Spend per Order: $ 145.01
Avg Spend per Order - Top 50%: $ 177.83
Avg Spend per Order - Top 25%: $ 189.84
Avg Spend per Order - Top 10%: $ 183.86


In [75]:
cluster_data = customers
cluster_data['Male'] = [1 if x == 'Male' else 0 for x in cluster_data['Gender']]


In [76]:
def avg_age(val):
    age_range = val['AgeGroup'].split('-')
    return (int(age_range[0]) + int(age_range[1]))/2

In [77]:
cluster_data['AgeClass'] = 0.0
for key, value in cluster_data.iterrows():
    cluster_data.at[key, 'AgeClass'] = avg_age(value)

In [81]:
categorical = pd.get_dummies(cluster_data['Acquisition'], prefix='Acquisition').astype(int)
cluster_enc = pd.concat([cluster_data[['CustomerID', 'Male', 'AgeClass']], categorical], axis=1)
cluster_enc

,CustomerID,Male,AgeClass,Acquisition_Email,Acquisition_Google,Acquisition_Meta,Acquisition_Organic
0,C10001,0,21.0,0,0,1,0
1,C10002,0,49.5,0,1,0,0
2,C10003,0,29.5,0,0,1,0
3,C10004,0,21.0,0,0,0,1
4,C10005,1,39.5,1,0,0,0
...,...,...,...,...,...,...,...
495,C10496,1,39.5,0,0,0,1
496,C10497,1,21.0,0,0,0,1
497,C10498,1,39.5,0,0,0,1
498,C10499,0,21.0,1,0,0,0


In [82]:
cluster_enc

,CustomerID,Male,AgeClass,Acquisition_Email,Acquisition_Google,Acquisition_Meta,Acquisition_Organic
0,C10001,0,21.0,0,0,1,0
1,C10002,0,49.5,0,1,0,0
2,C10003,0,29.5,0,0,1,0
3,C10004,0,21.0,0,0,0,1
4,C10005,1,39.5,1,0,0,0
...,...,...,...,...,...,...,...
495,C10496,1,39.5,0,0,0,1
496,C10497,1,21.0,0,0,0,1
497,C10498,1,39.5,0,0,0,1
498,C10499,0,21.0,1,0,0,0
